In [ ]:
import json
import os
from datetime import datetime


class ExpenseTracker:
    VALID_SORT_KEYS = {"date", "amount", "category"}

    def __init__(self, filename="expenses.json"):
        self.filename = filename
        self.expenses = self.load_expenses()

    # ------------------------------------------------------------------ I/O --

    def load_expenses(self):
        if not os.path.exists(self.filename):
            return []
        try:
            with open(self.filename, "r") as f:
                data = json.load(f)
            if not isinstance(data, list):
                raise ValueError("Expected a list at the top level.")
            return data
        except (json.JSONDecodeError, ValueError) as e:
            print(f"Warning: could not load '{self.filename}' ({e}). Starting fresh.")
            return []

    def save_expenses(self):
        with open(self.filename, "w") as f:
            json.dump(self.expenses, f, indent=4)

    # ------------------------------------------------------- input helpers ---

    @staticmethod
    def _prompt(label, allow_empty=False):
        while True:
            value = input(f"{label}: ").strip()
            if value or allow_empty:
                return value
            print("  This field cannot be empty.")

    @staticmethod
    def _prompt_amount():
        while True:
            raw = input("Amount (R): ").strip()
            try:
                amount = float(raw)
                if amount <= 0:
                    print("  Amount must be greater than zero.")
                else:
                    return amount
            except ValueError:
                print("  Please enter a valid number.")

    @staticmethod
    def _prompt_date(label="Date (YYYY-MM-DD, leave blank for today)"):
        while True:
            raw = input(f"{label}: ").strip()
            if not raw:
                return datetime.now().strftime("%Y-%m-%d")
            try:
                datetime.strptime(raw, "%Y-%m-%d")
                return raw
            except ValueError:
                print("  Use the format YYYY-MM-DD, e.g. 2025-03-15.")

    

    def add_expense(self):
        print("\n-- Add Expense --")
        category    = self._prompt("Category")
        description = self._prompt("Description")
        amount      = self._prompt_amount()
        date        = self._prompt_date()

        self.expenses.append({
            "id":          self._next_id(),
            "date":        date,
            "category":    category,
            "description": description,
            "amount":      amount,
        })
        self.save_expenses()
        print(f"  Saved: {description} — R{amount:.2f} on {date}.")

    def _next_id(self):
        return max((e.get("id", 0) for e in self.expenses), default=0) + 1

    def view_expenses(self, expenses=None, title="Expenses"):
        rows = expenses if expenses is not None else self.expenses
        if not rows:
            print("\n  No expenses to display.")
            return

        sort_by = input("Sort by (date / amount / category, default date): ").strip().lower()
        if sort_by not in self.VALID_SORT_KEYS:
            sort_by = "date"
        reverse = sort_by == "amount"
        rows = sorted(rows, key=lambda e: e.get(sort_by, ""), reverse=reverse)

        col = "{:<5} {:<12} {:<18} {:<28} {:>10}"
        header = col.format("ID", "Date", "Category", "Description", "Amount")
        print(f"\n-- {title} --")
        print(header)
        print("-" * len(header))
        for e in rows:
            desc = e["description"][:27] + "…" if len(e["description"]) > 27 else e["description"]
            cat  = e["category"][:17]   + "…" if len(e["category"]) > 17   else e["category"]
            print(col.format(
                e.get("id", "-"),
                e.get("date", ""),
                cat,
                desc,
                f"R{e['amount']:.2f}",
            ))
        print(f"\n  Total: R{sum(e['amount'] for e in rows):.2f}  ({len(rows)} item(s))")

    def delete_expense(self):
        print("\n-- Delete Expense --")
        self.view_expenses()
        if not self.expenses:
            return
        raw = input("\nEnter ID to delete (or blank to cancel): ").strip()
        if not raw:
            return
        try:
            target_id = int(raw)
        except ValueError:
            print("  Invalid ID.")
            return
        before = len(self.expenses)
        self.expenses = [e for e in self.expenses if e.get("id") != target_id]
        if len(self.expenses) < before:
            self.save_expenses()
            print(f"  Expense #{target_id} deleted.")
        else:
            print(f"  No expense with ID {target_id} found.")

    def filter_expenses(self):
        print("\n-- Filter Expenses --")
        print("  1. By category")
        print("  2. By date range")
        choice = input("Choose filter type: ").strip()

        if choice == "1":
            cats = sorted({e["category"] for e in self.expenses})
            if not cats:
                print("  No categories found.")
                return
            print("  Categories: " + ", ".join(cats))
            cat = self._prompt("Category")
            filtered = [e for e in self.expenses if e["category"].lower() == cat.lower()]
            self.view_expenses(filtered, title=f"Category: {cat}")

        elif choice == "2":
            start = self._prompt_date("Start date (YYYY-MM-DD, leave blank for today)")
            end   = self._prompt_date("End date   (YYYY-MM-DD, leave blank for today)")
            if start > end:
                start, end = end, start
            filtered = [e for e in self.expenses if start <= e.get("date", "") <= end]
            self.view_expenses(filtered, title=f"{start} → {end}")

        else:
            print("  Invalid choice.")

    def total_expenses(self):
        total = sum(e["amount"] for e in self.expenses)
        print(f"\n  Total across all expenses: R{total:.2f}")

    def category_report(self):
        if not self.expenses:
            print("\n  No expenses to report on.")
            return
        report = {}
        for e in self.expenses:
            report[e["category"]] = report.get(e["category"], 0) + e["amount"]

        total = sum(report.values())
        col   = "{:<25} {:>10}  {:>6}"
        print(f"\n-- Category Report --")
        print(col.format("Category", "Amount", "Share"))
        print("-" * 45)
        for cat, amt in sorted(report.items(), key=lambda x: x[1], reverse=True):
            pct = (amt / total * 100) if total else 0
            print(col.format(cat[:24], f"R{amt:.2f}", f"{pct:.1f}%"))
        print("-" * 45)
        print(col.format("TOTAL", f"R{total:.2f}", "100.0%"))

    def monthly_summary(self):
        if not self.expenses:
            print("\n  No expenses to summarise.")
            return
        by_month = {}
        for e in self.expenses:
            month = e.get("date", "")[:7]  # "YYYY-MM"
            by_month[month] = by_month.get(month, 0) + e["amount"]

        col = "{:<10} {:>12}"
        print("\n-- Monthly Summary --")
        print(col.format("Month", "Total"))
        print("-" * 24)
        for month in sorted(by_month):
            print(col.format(month, f"R{by_month[month]:.2f}"))

    def menu(self):
        options = {
            "1": ("Add expense",       self.add_expense),
            "2": ("View all expenses", self.view_expenses),
            "3": ("Filter expenses",   self.filter_expenses),
            "4": ("Delete an expense", self.delete_expense),
            "5": ("Total spent",       self.total_expenses),
            "6": ("Category report",   self.category_report),
            "7": ("Monthly summary",   self.monthly_summary),
            "8": ("Exit",              None),
        }

        while True:
            print("\n===== EXPENSE TRACKER =====")
            for key, (label, _) in options.items():
                print(f"  {key}. {label}")
            choice = input("Choose an option: ").strip()

            if choice == "8":
                print("Goodbye!")
                break
            elif choice in options:
                options[choice][1]()
            else:
                print("  Invalid choice — enter a number from 1 to 8.")


if __name__ == "__main__":
    tracker = ExpenseTracker()
    tracker.menu()